# Test a trained synthesis Unet2D
Loads `trained_nets/brats/Synth_T1ce/config.json` + its checkpoint, runs the validation
loader, and shows predicted T1ce vs ground truth with PSNR/SSIM. Uses your repo's
`build_model` / `build_loader` / `load_ckpt` so it matches training/saving exactly.

In [ ]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = '/home/ee2178/scratch/ee2178/ImMAP'    # so models/ datasets/ training/ import
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)

CONFIG_PATH = 'trained_nets/brats/Synth_T1ce/config.json'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Config

In [ ]:
with open(CONFIG_PATH) as f:
    cfg = json.load(f)
print('task :', cfg['task'])
print('model:', cfg['model']['type'], cfg['model']['params'])
print('val  :', {k: cfg['data']['val'][k] for k in ('root','input_idx','target_idx')})

ckpt_path = cfg['paths'].get('ckpt') or os.path.join(cfg['paths']['save_dir'], 'net.ckpt')
print('ckpt :', ckpt_path)

## Build model + load checkpoint

In [ ]:
import datasets                       # registers loaders (incl. 'synthesis')
from datasets.registry import build_loader
from models import build_model
from training.common import load_ckpt

model = build_model(cfg).to(device)
model, _, _, step = load_ckpt(ckpt_path, model=model, device=device)
model.eval()
print('loaded checkpoint at step', step)

## Validation data (one batch)

In [ ]:
cfg['data']['val']['num_workers'] = 0      # safer in a notebook
val_loader = build_loader(cfg['data']['val'], shuffle=False, drop_last=False)
print('val batches:', len(val_loader))

X, y = next(iter(val_loader))              # X (B,3,H,W) T1,T2,FLAIR ; y (B,1,H,W) T1ce
X, y = X.to(device), y.to(device)
with torch.no_grad():
    pred = model(X)
print('X', tuple(X.shape), '| y', tuple(y.shape), '| pred', tuple(pred.shape))

## Helpers: brain mask (mirrors training) + display scaling + metrics

In [ ]:
def brain_mask(x, eps=1e-6):
    bg = x.amin(dim=(-1, -2), keepdim=True)
    return (x > bg + eps).any(dim=1, keepdim=True).float()   # (B,1,H,W)

def show_scaled(ax, sl, m=None, title=''):
    sl = sl.detach().cpu().numpy()
    if m is not None and m.any():
        vmin, vmax = np.percentile(sl[m.detach().cpu().numpy().astype(bool)], [1, 99])
    else:
        vmin, vmax = np.percentile(sl, [1, 99])
    ax.imshow(sl, cmap='gray', vmin=vmin, vmax=vmax); ax.set_title(title); ax.axis('off')

def masked_psnr(p, t, m):
    mb = m.bool()
    if mb.sum() == 0: return float('nan')
    mse = (((p - t) ** 2)[mb]).mean()
    R = t[mb].max() - t[mb].min()
    return float(10 * torch.log10(R ** 2 / (mse + 1e-12)))

try:
    from skimage.metrics import structural_similarity as _ssim
    def ssim2d(p, t):
        p = p.detach().cpu().numpy(); t = t.detach().cpu().numpy()
        return float(_ssim(t, p, data_range=t.max() - t.min()))
except Exception:
    def ssim2d(p, t): return float('nan')
    print('skimage not available -> SSIM skipped')

## Qualitative: input / GT / prediction / error

In [ ]:
k = min(4, X.shape[0])
m = brain_mask(X)
fig, axes = plt.subplots(k, 4, figsize=(14, 3.4 * k))
axes = np.atleast_2d(axes)
for i in range(k):
    mi = m[i, 0]
    show_scaled(axes[i, 0], X[i, 0], mi, 'Input T1')
    show_scaled(axes[i, 1], y[i, 0], mi, 'GT T1ce')
    show_scaled(axes[i, 2], pred[i, 0], mi, 'Predicted')
    err = ((pred[i, 0] - y[i, 0]).abs() * mi).detach().cpu().numpy()
    axes[i, 3].imshow(err, cmap='magma'); axes[i, 3].set_title(
        f'|err|  PSNR={masked_psnr(pred[i], y[i], m[i]):.1f}dB  SSIM={ssim2d(pred[i,0], y[i,0]):.3f}')
    axes[i, 3].axis('off')
plt.tight_layout(); plt.show()

## Quantitative: metrics over the validation set

In [ ]:
N_BATCHES = 20      # cap for speed; raise to len(val_loader) for the full set
psnrs, ssims = [], []
with torch.no_grad():
    for bi, (Xv, yv) in enumerate(val_loader):
        if bi >= N_BATCHES: break
        Xv, yv = Xv.to(device), yv.to(device)
        pv = model(Xv); mv = brain_mask(Xv)
        for i in range(Xv.shape[0]):
            psnrs.append(masked_psnr(pv[i], yv[i], mv[i]))
            ssims.append(ssim2d(pv[i, 0], yv[i, 0]))
psnrs = np.array(psnrs); ssims = np.array(ssims)
print(f'slices evaluated : {len(psnrs)}')
print(f'PSNR (masked)    : {np.nanmean(psnrs):.2f} +/- {np.nanstd(psnrs):.2f} dB')
print(f'SSIM             : {np.nanmean(ssims):.3f} +/- {np.nanstd(ssims):.3f}')